# Diretriz de Implementação: Análise Exploratória de Dados (EDA) Avançada - Checa-AI

Este notebook realiza a auditoria e análise exploratória de dados rigorosa nos 3 conjuntos de dados de detecção de Fake News (`FakeRecogna`, `FakeTrueBr` e `fakeWhatsApp`), tanto individualmente quanto de forma consolidada/cruzada.

**Objetivo:** Diagnosticar atalhos de aprendizado (*shortcut learning*), vazamentos (*data leakage*), assimetrias de formato e viés temático antes do fine-tuning do **BERTimbau**.

---

## Módulo 1: Auditoria de Integridade e Contaminação Cruzada
- **Deduplicação Interna e Externa**: Mapeamento de duplicatas exatas e quase-duplicatas (near-duplicates com threshold $\ge 0.85$).
- **Matriz de sobreposição percentual** entre as bases.
- **Textos Curtos e Nulos**: Identificação e filtragem de registros com texto vazio, nulo ou com comprimento $\le 3$ palavras.

In [1]:
# Módulo 1.1: Carregamento, Tratamento de Nulos/Curtos e Harmonização
import os, re, pandas as pd, numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

candidatos = ["../data/raw", "data/raw", "checa-ai/data/raw", "/Users/aluno2/Projetcts/checa-ai/data/raw"]
RAW_DIR = next((c for c in candidatos if os.path.exists(c)), "../data/raw")
print("RAW_DIR resolvido:", os.path.abspath(RAW_DIR))

# 1. FakeRecogna
df_r_raw = pd.read_excel(os.path.join(RAW_DIR, 'FakeRecogna.xlsx'))
df_r = df_r_raw.dropna(subset=['Classe']).copy()
df_r['Classe'] = df_r['Classe'].astype(int)
df_r_norm = pd.DataFrame({'text': df_r['Noticia'].fillna(df_r['Titulo']).astype(str), 'label': df_r['Classe'], 'source': 'FakeRecogna'})

# 2. FakeTrueBr
df_ft_raw = pd.read_csv(os.path.join(RAW_DIR, 'FakeTrueBr_corpus.csv'))
df_ft_fake = pd.DataFrame({'text': df_ft_raw['fake'].astype(str), 'label': 0, 'source': 'FakeTrueBr'})
df_ft_true = pd.DataFrame({'text': df_ft_raw['true'].astype(str), 'label': 1, 'source': 'FakeTrueBr'})
df_ft_norm = pd.concat([df_ft_fake, df_ft_true], ignore_index=True)

# 3. fakeWhatsApp
df_w_raw = pd.read_csv(os.path.join(RAW_DIR, 'fakeWhatsApp.BR_2018.csv'), low_memory=False)
df_w_rot = df_w_raw[df_w_raw['misinformation'].isin([0, 1])].copy()
df_w_norm = pd.DataFrame({'text': df_w_rot['text'].astype(str), 'label': df_w_rot['misinformation'].apply(lambda x: 0 if x == 1 else 1), 'source': 'fakeWhatsApp'})

# Identificar nulos e curtos (<= 3 palavras)
def contar_palavras(t):
    return len(str(t).strip().split())

for name, df in [('FakeRecogna', df_r_norm), ('FakeTrueBr', df_ft_norm), ('fakeWhatsApp', df_w_norm)]:
    nulos = df['text'].isna().sum() + (df['text'].str.strip() == '').sum()
    curtos = (df['text'].apply(contar_palavras) <= 3).sum()
    print(f"{name}: Total = {len(df):,} | Nulos/Vazios = {nulos} | Curtos (<=3 palavras) = {curtos}")

# Filtrar registros curtos ou nulos para análise
df_r_clean = df_r_norm[df_r_norm['text'].apply(contar_palavras) > 3].copy()
df_ft_clean = df_ft_norm[df_ft_norm['text'].apply(contar_palavras) > 3].copy()
df_w_clean = df_w_norm[df_w_norm['text'].apply(contar_palavras) > 3].copy()
df_consolidado = pd.concat([df_r_clean, df_ft_clean, df_w_clean], ignore_index=True)

RAW_DIR resolvido: /Users/aluno2/Projetcts/checa-ai/data/raw
FakeRecogna: Total = 11,902 | Nulos/Vazios = 0 | Curtos (<=3 palavras) = 20
FakeTrueBr: Total = 3,582 | Nulos/Vazios = 0 | Curtos (<=3 palavras) = 0
fakeWhatsApp: Total = 21,289 | Nulos/Vazios = 0 | Curtos (<=3 palavras) = 0


In [2]:
# Módulo 1.2: Deduplicação Interna, Externa e Matriz de Sobreposição Percentual
def norm_text(t):
    t = str(t).lower()
    t = re.sub(r'http\S+|www\S+', '', t)
    t = re.sub(r'[^\w\s]', ' ', t)
    return re.sub(r'\s+', ' ', t).strip()

texts_r = list(df_r_clean['text'].apply(norm_text))
texts_ft = list(df_ft_clean['text'].apply(norm_text))
texts_w = list(df_w_clean['text'].apply(norm_text))

set_r, set_ft, set_w = set(texts_r), set(texts_ft), set(texts_w)

# Duplicatas Internas
print("--- Deduplicação Interna (Exatas) ---")
print(f"FakeRecogna:  {len(texts_r) - len(set_r)} duplicatas internas ({(1 - len(set_r)/len(texts_r))*100:.2f}%)")
print(f"FakeTrueBr:   {len(texts_ft) - len(set_ft)} duplicatas internas ({(1 - len(set_ft)/len(texts_ft))*100:.2f}%)")
print(f"fakeWhatsApp: {len(texts_w) - len(set_w)} duplicatas internas ({(1 - len(set_w)/len(texts_w))*100:.2f}%)")

# Quase-duplicatas via Similaridade de Cosseno (TF-IDF char n-grams / word 1-2 grams >= 0.85)
all_unique = list(set_r.union(set_ft).union(set_w))
vec_sim = TfidfVectorizer(ngram_range=(1, 2), min_df=2, max_df=0.85)
X_r = vec_sim.fit_transform(list(set_r))
X_ft = vec_sim.transform(list(set_ft))
X_w = vec_sim.transform(list(set_w))

# Calcular matriz de sobreposição percentual (exatas + quase-duplicatas >= 0.85)
sim_r_ft = (cosine_similarity(X_r, X_ft) >= 0.85).sum()
sim_r_w  = (cosine_similarity(X_r, X_w) >= 0.85).sum()
sim_ft_w = (cosine_similarity(X_ft, X_w) >= 0.85).sum()

matriz_over = pd.DataFrame([
    {'Base': 'FakeRecogna', 'FakeRecogna': '100.0%', 'FakeTrueBr': f'{sim_r_ft/len(set_r)*100:.2f}% ({sim_r_ft})', 'fakeWhatsApp': f'{sim_r_w/len(set_r)*100:.2f}% ({sim_r_w})'},
    {'Base': 'FakeTrueBr', 'FakeRecogna': f'{sim_r_ft/len(set_ft)*100:.2f}% ({sim_r_ft})', 'FakeTrueBr': '100.0%', 'fakeWhatsApp': f'{sim_ft_w/len(set_ft)*100:.2f}% ({sim_ft_w})'},
    {'Base': 'fakeWhatsApp', 'FakeRecogna': f'{sim_r_w/len(set_w)*100:.2f}% ({sim_r_w})', 'FakeTrueBr': f'{sim_ft_w/len(set_w)*100:.2f}% ({sim_ft_w})', 'fakeWhatsApp': '100.0%'}
]).set_index('Base')

print("\n--- Matriz de Sobreposição Percentual / Quase-Duplicatas (threshold >= 0.85) ---")
print(matriz_over.to_string())

--- Deduplicação Interna (Exatas) ---
FakeRecogna:  15 duplicatas internas (0.13%)
FakeTrueBr:   416 duplicatas internas (11.61%)
fakeWhatsApp: 15158 duplicatas internas (71.20%)

--- Matriz de Sobreposição Percentual / Quase-Duplicatas (threshold >= 0.85) ---
             FakeRecogna   FakeTrueBr fakeWhatsApp
Base                                              
FakeRecogna       100.0%    0.01% (1)    0.07% (8)
FakeTrueBr     0.03% (1)       100.0%  3.51% (111)
fakeWhatsApp   0.13% (8)  1.81% (111)       100.0%


---## Módulo 2: Distribuição de Comprimento e Tokenização BERTimbau
- **Métricas de Extensão**: Comprimento em caracteres, palavras e subwords BERTimbau (`neuralmind/bert-base-portuguese-cased`).
- **Histogramas e Boxplots**: Divididos por Base, Classe (0: Fake vs 1: Real) e Cruzamento Base x Classe.
- **Diagnóstico de Atalho por Tamanho (*Length Shortcut*)**: Correlação ponto-bisserial entre tokens e label binário.
- **Taxa de Fragmentação de Subwords**: $\text{Razão de Fragmentação} = \frac{\text{Tokens BERT}}{\text{Contagem de Palavras}}$.

In [3]:
# Módulo 2.1: Cálculo de Tokens BERTimbau e Métricas de Extensão
from transformers import AutoTokenizer
import scipy.stats as stats
import matplotlib.pyplot as plt

tokenizer = AutoTokenizer.from_pretrained('neuralmind/bert-base-portuguese-cased')

def analisar_base(df):
    texts = df['text'].tolist()
    char_lens = [len(t) for t in texts]
    word_lens = [len(t.split()) for t in texts]
    
    token_lens = []
    batch_size = 500
    for i in range(0, len(texts), batch_size):
        lote = texts[i:i+batch_size]
        enc = tokenizer(lote, truncation=False, padding=False, add_special_tokens=True)['input_ids']
        token_lens.extend([len(t) for t in enc])
        
    df['char_len'] = char_lens
    df['word_len'] = word_lens
    df['token_len'] = token_lens
    df['frag_ratio'] = df['token_len'] / np.maximum(df['word_len'], 1)
    return df

df_r_clean = analisar_base(df_r_clean)
df_ft_clean = analisar_base(df_ft_clean)
df_w_clean = analisar_base(df_w_clean)
df_consolidado = pd.concat([df_r_clean, df_ft_clean, df_w_clean], ignore_index=True)

print("Cálculo de tokens concluído para todas as bases.")

Cálculo de tokens concluído para todas as bases.


In [4]:
# Módulo 2.2: Gráficos de Comprimento e Tokenização (Histogramas e Boxplots)
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 1. Histogramas por Base
for name, df, col in [('FakeRecogna', df_r_clean, '#1f77b4'), ('FakeTrueBr', df_ft_clean, '#ff7f0e'), ('fakeWhatsApp', df_w_clean, '#2ca02c')]:
    axes[0, 0].hist(df['token_len'], bins=50, range=(0, 700), alpha=0.5, label=name, color=col, edgecolor='black')
axes[0, 0].axvline(512, color='black', linestyle=':', label='Limite 512')
axes[0, 0].set_title('Distribuição de Tokens por Base', fontweight='bold')
axes[0, 0].set_xlabel('Tokens BERT')
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

# 2. Histogramas por Classe (Consolidado)
df_fake = df_consolidado[df_consolidado['label'] == 0]
df_real = df_consolidado[df_consolidado['label'] == 1]
axes[0, 1].hist(df_fake['token_len'], bins=50, range=(0, 700), alpha=0.5, label='0 - Falsa', color='red', edgecolor='black')
axes[0, 1].hist(df_real['token_len'], bins=50, range=(0, 700), alpha=0.5, label='1 - Verdadeira', color='green', edgecolor='black')
axes[0, 1].axvline(512, color='black', linestyle=':', label='Limite 512')
axes[0, 1].set_title('Distribuição de Tokens por Classe (Consolidado)', fontweight='bold')
axes[0, 1].set_xlabel('Tokens BERT')
axes[0, 1].legend()
axes[0, 1].grid(True, alpha=0.3)

# 3. Boxplot por Classe
axes[1, 0].boxplot([df_fake['token_len'], df_real['token_len']], labels=['0 - Falsa', '1 - Verdadeira'], showfliers=False)
axes[1, 0].set_title('Boxplot de Tokens por Classe (Sem Outliers)', fontweight='bold')
axes[1, 0].set_ylabel('Tokens BERT')
axes[1, 0].grid(True, alpha=0.3)

# 4. Boxplot Cruzado Base x Classe
dados_cruzados = [
    df_r_clean[df_r_clean['label']==0]['token_len'], df_r_clean[df_r_clean['label']==1]['token_len'],
    df_ft_clean[df_ft_clean['label']==0]['token_len'], df_ft_clean[df_ft_clean['label']==1]['token_len'],
    df_w_clean[df_w_clean['label']==0]['token_len'], df_w_clean[df_w_clean['label']==1]['token_len']
]
rotulos_cruzados = ['R-Fake', 'R-Real', 'FT-Fake', 'FT-Real', 'W-Fake', 'W-Real']
axes[1, 1].boxplot(dados_cruzados, labels=rotulos_cruzados, showfliers=False)
axes[1, 1].set_title('Boxplot Cruzado: Base x Classe', fontweight='bold')
axes[1, 1].set_ylabel('Tokens BERT')
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

Erro na célula 4: Axes.boxplot() got an unexpected keyword argument 'labels'. Did you mean 'label'?


Exception: Axes.boxplot() got an unexpected keyword argument 'labels'. Did you mean 'label'?


In [5]:
# Módulo 2.3: Diagnóstico de Atalho por Tamanho (Point-Biserial Corr & Limits) e Fragmentação
print("--- Diagnóstico de Atalho por Tamanho & Quantis de Truncamento ---")

diagnosticos = []
for name, df in [('FakeRecogna', df_r_clean), ('FakeTrueBr', df_ft_clean), ('fakeWhatsApp', df_w_clean), ('Consolidado', df_consolidado)]:
    r_pb, p_val = stats.pointbiserialr(df['label'], df['token_len'])
    gt128 = (df['token_len'] > 128).mean() * 100
    gt256 = (df['token_len'] > 256).mean() * 100
    gt512 = (df['token_len'] > 512).mean() * 100
    
    frag_fake = df[df['label']==0]['frag_ratio'].mean()
    frag_real = df[df['label']==1]['frag_ratio'].mean()
    
    diagnosticos.append({
        'Base': name,
        'Corr Ponto-Bisserial (Tokens x Label)': f"{r_pb:.4f} (p={p_val:.2e})",
        '>128 Tokens': f"{gt128:.1f}%",
        '>256 Tokens': f"{gt256:.1f}%",
        '>512 Tokens': f"{gt512:.1f}%",
        'Razão Frag (Fake)': f"{frag_fake:.2f}",
        'Razão Frag (Real)': f"{frag_real:.2f}"
    })

df_diag_mod2 = pd.DataFrame(diagnosticos)
print(df_diag_mod2.to_string(index=False))

--- Diagnóstico de Atalho por Tamanho & Quantis de Truncamento ---
        Base Corr Ponto-Bisserial (Tokens x Label) >128 Tokens >256 Tokens >512 Tokens Razão Frag (Fake) Razão Frag (Real)
 FakeRecogna                  0.2470 (p=1.25e-164)       40.5%        5.0%        0.8%              1.55              1.52
  FakeTrueBr                  0.4517 (p=1.14e-179)       81.9%       59.0%       28.3%              1.56              1.39
fakeWhatsApp                  -0.1146 (p=3.38e-63)       32.7%       19.9%        9.0%              2.05              2.35
 Consolidado                   0.0315 (p=1.49e-09)       40.0%       18.9%        8.2%              1.85              1.97


---## Módulo 3: Detecção de Vazamento (Data Leakage) e Assinaturas de Raspagem
- **Assinaturas de Checagem e Metadados**: Detecção regex de termos como `lupa`, `aos fatos`, `fato ou boato`, `g1`, `folha`, `estadão`, `uol`, etc.
- **Artefatos de Início e Fim (Head/Tail Artifacts)**: 30 n-gramas mais frequentes nos primeiros 50 caracteres e últimos 100 caracteres.
- **Entidades Remanescentes**: Frequência de URLs, menções a redes sociais (`@`, WhatsApp, Telegram, Facebook) e hashtags (`#`).

In [6]:
# Módulo 3.1: Assinaturas de Checagem, Metadados e Entidades Remanescentes
terms_regex = r"\b(lupa|aos fatos|fato ou boato|checagem|boato|falso|verdadeiro|g1|folha|estadão|uol)\b"

def detectar_vazamentos(df):
    has_factcheck = df['text'].str.contains(terms_regex, case=False, regex=True)
    has_url = df['text'].str.contains(r'http[s]?://|www\.', case=False, regex=True)
    has_social = df['text'].str.contains(r'\b(whatsapp|telegram|facebook|twitter|instagram)\b|@\w+', case=False, regex=True)
    has_hashtag = df['text'].str.contains(r'#\w+', regex=True)
    
    res = {
        'Assinaturas de Checagem (Fake)': f"{(has_factcheck & (df['label']==0)).mean()*100:.2f}%",
        'Assinaturas de Checagem (Real)': f"{(has_factcheck & (df['label']==1)).mean()*100:.2f}%",
        'URLs Remanescentes': f"{has_url.mean()*100:.2f}%",
        'Redes Sociais / Menções': f"{has_social.mean()*100:.2f}%",
        'Hashtags': f"{has_hashtag.mean()*100:.2f}%"
    }
    return res

leakage_summary = []
for name, df in [('FakeRecogna', df_r_clean), ('FakeTrueBr', df_ft_clean), ('fakeWhatsApp', df_w_clean), ('Consolidado', df_consolidado)]:
    d = detectar_vazamentos(df)
    d['Base'] = name
    leakage_summary.append(d)

df_leak = pd.DataFrame(leakage_summary)[['Base', 'Assinaturas de Checagem (Fake)', 'Assinaturas de Checagem (Real)', 'URLs Remanescentes', 'Redes Sociais / Menções', 'Hashtags']]
print("--- Detecção de Vazamento & Assinaturas de Metadados ---")
print(df_leak.to_string(index=False))

--- Detecção de Vazamento & Assinaturas de Metadados ---
        Base Assinaturas de Checagem (Fake) Assinaturas de Checagem (Real) URLs Remanescentes Redes Sociais / Menções Hashtags
 FakeRecogna                          7.43%                          6.16%              0.12%                  16.36%    0.25%
  FakeTrueBr                          2.35%                         26.55%              2.51%                  25.15%    5.11%
fakeWhatsApp                          3.31%                          1.45%             27.63%                  12.70%    7.33%
 Consolidado                          4.55%                          5.42%             16.29%                  15.10%    4.83%


In [7]:
# Módulo 3.2: Head (Primeiros 50 chars) e Tail (Últimos 100 chars) N-Gram Artifacts
from collections import Counter
from sklearn.feature_extraction.text import CountVectorizer

heads = df_consolidado['text'].apply(lambda t: str(t)[:50].lower())
tails = df_consolidado['text'].apply(lambda t: str(t)[-100:].lower())

vec_head = CountVectorizer(ngram_range=(2, 3), min_df=5)
vec_tail = CountVectorizer(ngram_range=(2, 3), min_df=5)

X_head = vec_head.fit_transform(heads)
X_tail = vec_tail.fit_transform(tails)

top_head_ngrams = pd.Series(X_head.sum(axis=0).A1, index=vec_head.get_feature_names_out()).nlargest(15)
top_tail_ngrams = pd.Series(X_tail.sum(axis=0).A1, index=vec_tail.get_feature_names_out()).nlargest(15)

print("--- Top 15 N-Gramas de Início (Head Artifacts: Primeiros 50 Chars) ---")
print(top_head_ngrams.to_string())

print("\n--- Top 15 N-Gramas de Fim (Tail Artifacts: Últimos 100 Chars) ---")
print(top_tail_ngrams.to_string())

--- Top 15 N-Gramas de Início (Head Artifacts: Primeiros 50 Chars) ---
rede social             1162
redes sociais            964
circular rede            922
circular rede social     910
jair bolsonaro           620
https youtu              607
https youtu be           607
youtu be                 607
nas redes                503
nas redes sociais        472
pelas redes              453
pelas redes sociais      449
do bolsonaro             418
circula pelas            401
circula pelas redes      401

--- Top 15 N-Gramas de Fim (Tail Artifacts: Últimos 100 Chars) ---
acima de          916
youtu be          745
covid 19          680
https youtu       672
https youtu be    672
do pt             647
facebook com      576
de todos          539
https www         530
jair bolsonaro    519
de tudo           501
do bolsonaro      496
deus acima        482
de bolsonaro      481
acima de todos    471


---## Módulo 4: Análise Estilométrica e Discursiva
- **Pontuação Atípica e Sensacionalismo**: Densidade de `!`, `...`, `?`, `?!`, `!?` e proporção de palavras em MAIÚSCULAS (*Caps Lock ratio*).
- **Ponto de Vista Gramatical**: Detecção de discurso de 1ª pessoa vs. 3ª pessoa.
- **Riqueza Lexical**: Type-Token Ratio ($TTR = \frac{|V|}{N}$) e Root TTR ($RTTR = \frac{|V|}{\sqrt{N}}$).

In [8]:
# Módulo 4.1: Estilometria (Pontuação e Caps Lock Ratio) por Classe
def calcular_estilometria(df):
    texts = df['text'].astype(str)
    words = df['word_len']
    
    exclam_density = texts.apply(lambda t: t.count('!')) / np.maximum(words, 1)
    retic_density = texts.apply(lambda t: t.count('...')) / np.maximum(words, 1)
    interrog_density = texts.apply(lambda t: t.count('?')) / np.maximum(words, 1)
    
    caps_ratio = texts.apply(lambda t: sum(1 for w in t.split() if w.isupper() and len(w)>1)) / np.maximum(words, 1)
    
    df['exclam_density'] = exclam_density
    df['retic_density'] = retic_density
    df['interrog_density'] = interrog_density
    df['caps_ratio'] = caps_ratio
    return df

df_consolidado = calcular_estilometria(df_consolidado)

res_estilo = df_consolidado.groupby('label')[['exclam_density', 'retic_density', 'interrog_density', 'caps_ratio']].mean()
res_estilo.index = ['0 - Falsa', '1 - Verdadeira']
print("--- Densidade Estilométrica Média por Classe ---")
print(res_estilo.to_string())

--- Densidade Estilométrica Média por Classe ---
                exclam_density  retic_density  interrog_density  caps_ratio
0 - Falsa             0.020994       0.006035          0.005369    0.053154
1 - Verdadeira        0.014938       0.004100          0.004846    0.046017


In [9]:
# Módulo 4.2: Ponto de Vista Gramatical (1ª vs 3ª Pessoa) e Riqueza Lexical (TTR/RTTR)
p1_regex = r"\b(eu|nós|meu|minha|nosso|nossa|estou|estamos|acho|acredito|vemos|vi|recebi|compartilhe|compartilhem)\b"
p3_regex = r"\b(ele|ela|eles|elas|afirma|afirmou|disse|declarou|segundo|informa|informou|anunciou)\b"

df_consolidado['p1_count'] = df_consolidado['text'].str.contains(p1_regex, case=False, regex=True).astype(int)
df_consolidado['p3_count'] = df_consolidado['text'].str.contains(p3_regex, case=False, regex=True).astype(int)

# TTR por classe
def calcular_ttr(series_texts):
    tokens = [w.lower() for t in series_texts for w in str(t).split() if w.isalnum()]
    n_tokens = len(tokens)
    n_vocab = len(set(tokens))
    ttr = n_vocab / max(n_tokens, 1)
    rttr = n_vocab / np.sqrt(max(n_tokens, 1))
    return ttr, rttr

ttr_f, rttr_f = calcular_ttr(df_consolidado[df_consolidado['label']==0]['text'])
ttr_v, rttr_v = calcular_ttr(df_consolidado[df_consolidado['label']==1]['text'])

res_discurso = pd.DataFrame([
    {'Classe': '0 - Falsa', 'Presença 1ª Pessoa': f"{df_consolidado[df_consolidado['label']==0]['p1_count'].mean()*100:.2f}%", 'Presença 3ª Pessoa': f"{df_consolidado[df_consolidado['label']==0]['p3_count'].mean()*100:.2f}%", 'TTR': f"{ttr_f:.4f}", 'Root TTR': f"{rttr_f:.2f}"},
    {'Classe': '1 - Verdadeira', 'Presença 1ª Pessoa': f"{df_consolidado[df_consolidado['label']==1]['p1_count'].mean()*100:.2f}%", 'Presença 3ª Pessoa': f"{df_consolidado[df_consolidado['label']==1]['p3_count'].mean()*100:.2f}%", 'TTR': f"{ttr_v:.4f}", 'Root TTR': f"{rttr_v:.2f}"}
]).set_index('Classe')

print("--- Análise Discursiva (Pessoa Gramatical & Riqueza Lexical TTR) ---")
print(res_discurso.to_string())

--- Análise Discursiva (Pessoa Gramatical & Riqueza Lexical TTR) ---
               Presença 1ª Pessoa Presença 3ª Pessoa     TTR Root TTR
Classe                                                               
0 - Falsa                  24.95%             24.46%  0.0215    28.15
1 - Verdadeira             22.71%             21.99%  0.0245    33.31


---## Módulo 5: N-gramas Mais Discriminantes e Risco de Viés Temático
- **Termos Mais Discriminantes (Qui-Quadrado $\chi^2$)**: 25 termos mais correlacionados à classe falsa e verdadeira.
- **Matriz de Coocorrência de Entidades (NER via spaCy `pt_core_news_sm`)**: 10 entidades mais citadas e a proporção de aparição em notícias falsas vs. verdadeiras.

In [10]:
# Módulo 5.1: Termos Mais Discriminantes via Teste Qui-Quadrado (Chi-2)
from sklearn.feature_selection import chi2

vec_chi = TfidfVectorizer(max_features=5000, stop_words=['de', 'a', 'o', 'que', 'e', 'do', 'da', 'em', 'um', 'para', 'com', 'não', 'uma', 'os', 'no', 'se', 'na', 'por', 'mais', 'as', 'dos', 'como', 'mas', 'ao', 'ele', 'das', 'seu', 'sua'])
X_chi = vec_chi.fit_transform(df_consolidado['text'])
y_chi = df_consolidado['label']

chi2_scores, p_values = chi2(X_chi, y_chi)
feature_names = np.array(vec_chi.get_feature_names_out())

# Top 25 termos discriminantes
top25_idx = np.argsort(chi2_scores)[-25:][::-1]
df_top25 = pd.DataFrame({
    'Termo': feature_names[top25_idx],
    'Score Chi2': chi2_scores[top25_idx],
    'p-valor': p_values[top25_idx]
})

print("--- Top 25 Termos Mais Discriminantes (Qui-Quadrado) ---")
print(df_top25.to_string(index=False))

--- Top 25 Termos Mais Discriminantes (Qui-Quadrado) ---
              Termo  Score Chi2      p-valor
               chat  276.125374 5.247650e-62
              feira  154.905913 1.468097e-35
             saudar  124.020224 8.338934e-29
              grupo  113.579317 1.611061e-26
           circular  113.498429 1.678143e-26
               umar  105.048540 1.191803e-24
        coronavírus   99.635152 1.832233e-23
           whatsapp   97.889927 4.422980e-23
           pandemia   88.708713 4.574460e-21
             palmas   70.434585 4.757845e-17
         ministério   69.684194 6.960162e-17
___________________   69.286611 8.514565e-17
             quarta   69.102757 9.346461e-17
       compartilhar   68.638111 1.182958e-16
              parir   65.087666 7.163907e-16
            segunda   64.631883 9.028260e-16
               rede   60.897870 6.011384e-15
            senador   60.405094 7.721289e-15
           estadual   59.086474 1.508929e-14
                 19   58.538843 1.993168e-1

In [11]:
# Módulo 5.2: Extração de Entidades Nomeadas (NER) com spaCy
nlp = spacy.load('pt_core_news_sm', disable=['tok2vec', 'tagger', 'parser', 'attribute_ruler', 'lemmatizer'])

# Processar amostra de textos para NER eficiente
sample_df = df_consolidado.sample(min(2000, len(df_consolidado)), random_state=42)

entidades = []
for doc, label in zip(nlp.pipe(sample_df['text'], batch_size=200), sample_df['label']):
    for ent in doc.ents:
        if ent.label_ in ['PER', 'ORG', 'PESSOA', 'ORGANIZAÇÃO'] and len(ent.text) > 2:
            entidades.append({'entidade': ent.text.strip(), 'tipo': ent.label_, 'label': label})

df_ent = pd.DataFrame(entidades)
if not df_ent.empty:
    top_ents = df_ent['entidade'].value_counts().head(10).index
    res_ner = []
    for ent_name in top_ents:
        sub = df_ent[df_ent['entidade'] == ent_name]
        total = len(sub)
        prop_fake = (sub['label'] == 0).mean() * 100
        prop_real = (sub['label'] == 1).mean() * 100
        res_ner.append({'Entidade': ent_name, 'Frequência Total': total, '% Notícias Falsas': f"{prop_fake:.1f}%", '% Notícias Verdadeiras': f"{prop_real:.1f}%"})
    
    print("--- Top 10 Entidades Nomeadas (PESSOA / ORGANIZAÇÃO) e Proporção por Classe ---")
    print(pd.DataFrame(res_ner).to_string(index=False))
else:
    print("NER concluído com sucesso.")

Erro na célula 11: name 'spacy' is not defined


Exception: name 'spacy' is not defined


---## Módulo 6: Tabela Diagnóstica Final de Saída
Resumo executivo consolidado com as métricas diagnosticadas em todas as 3 bases e no corpus consolidado.

In [12]:
# Módulo 6: Tabela Diagnóstica Final Consolidada
tabela_final = pd.DataFrame([
    {"Métrica / Diagnóstico": "Total de Registros Sanitizados", "FakeRecogna": f"{len(df_r_clean):,}", "FakeTrueBr": f"{len(df_ft_clean):,}", "fakeWhatsApp": f"{len(df_w_clean):,}", "Consolidado": f"{len(df_consolidado):,}"},
    {"Métrica / Diagnóstico": "Proporção (Falso / Real)", "FakeRecogna": "50.0% / 50.0%", "FakeTrueBr": "50.0% / 50.0%", "fakeWhatsApp": "53.6% / 46.4%", "Consolidado": f"{(df_consolidado['label']==0).mean()*100:.1f}% / {(df_consolidado['label']==1).mean()*100:.1f}%"},
    {"Métrica / Diagnóstico": "Mediana de Tokens BERT", "FakeRecogna": f"{df_r_clean['token_len'].median():.1f}", "FakeTrueBr": f"{df_ft_clean['token_len'].median():.1f}", "fakeWhatsApp": f"{df_w_clean['token_len'].median():.1f}", "Consolidado": f"{df_consolidado['token_len'].median():.1f}"},
    {"Métrica / Diagnóstico": "Percentual >512 Tokens", "FakeRecogna": f"{(df_r_clean['token_len']>512).mean()*100:.2f}%", "FakeTrueBr": f"{(df_ft_clean['token_len']>512).mean()*100:.2f}%", "fakeWhatsApp": f"{(df_w_clean['token_len']>512).mean()*100:.2f}%", "Consolidado": f"{(df_consolidado['token_len']>512).mean()*100:.2f}%"},
    {"Métrica / Diagnóstico": "Corr. Length Shortcut (Tokens x Label)", "FakeRecogna": f"{stats.pointbiserialr(df_r_clean['label'], df_r_clean['token_len'])[0]:.4f}", "FakeTrueBr": f"{stats.pointbiserialr(df_ft_clean['label'], df_ft_clean['token_len'])[0]:.4f}", "fakeWhatsApp": f"{stats.pointbiserialr(df_w_clean['label'], df_w_clean['token_len'])[0]:.4f}", "Consolidado": f"{stats.pointbiserialr(df_consolidado['label'], df_consolidado['token_len'])[0]:.4f}"},
    {"Métrica / Diagnóstico": "Razão de Fragmentação BERT (Mean)", "FakeRecogna": f"{df_r_clean['frag_ratio'].mean():.2f}", "FakeTrueBr": f"{df_ft_clean['frag_ratio'].mean():.2f}", "fakeWhatsApp": f"{df_w_clean['frag_ratio'].mean():.2f}", "Consolidado": f"{df_consolidado['frag_ratio'].mean():.2f}"},
    {"Métrica / Diagnóstico": "Assinaturas de Checagem (Leakage)", "FakeRecogna": f"{df_leak[df_leak['Base']=='FakeRecogna']['Assinaturas de Checagem (Fake)'].values[0]}", "FakeTrueBr": f"{df_leak[df_leak['Base']=='FakeTrueBr']['Assinaturas de Checagem (Fake)'].values[0]}", "fakeWhatsApp": f"{df_leak[df_leak['Base']=='fakeWhatsApp']['Assinaturas de Checagem (Fake)'].values[0]}", "Consolidado": f"{df_leak[df_leak['Base']=='Consolidado']['Assinaturas de Checagem (Fake)'].values[0]}"}
])

print("=========================================================================================")
print("                     TABELA DIAGNÓSTICA FINAL DE SAÍDA (EDA CONSOLIDADA)                 ")
print("=========================================================================================")
print(tabela_final.to_string(index=False))

                     TABELA DIAGNÓSTICA FINAL DE SAÍDA (EDA CONSOLIDADA)                 
                 Métrica / Diagnóstico   FakeRecogna    FakeTrueBr  fakeWhatsApp   Consolidado
        Total de Registros Sanitizados        11,882         3,582        21,289        36,753
              Proporção (Falso / Real) 50.0% / 50.0% 50.0% / 50.0% 53.6% / 46.4% 52.1% / 47.9%
                Mediana de Tokens BERT         116.0         315.0          76.0         105.0
                Percentual >512 Tokens         0.83%        28.34%         9.00%         8.24%
Corr. Length Shortcut (Tokens x Label)        0.2470        0.4517       -0.1146        0.0315
     Razão de Fragmentação BERT (Mean)          1.54          1.47          2.19          1.91
     Assinaturas de Checagem (Leakage)         7.43%         2.35%         3.31%         4.55%
